 # RAG / ChromaDB (Response Planner's memory)

In [3]:
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
import os

load_dotenv() 

client = chromadb.PersistentClient(path="./chroma_dev")

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.environ.get("OPENAI_API_KEY"),
    model_name="text-embedding-3-small",
)

collection = client.get_or_create_collection(
    name="disaster_tweets",
    embedding_function=openai_ef,
)


In [4]:
sample_protocols = [
    {
        "id": "flood_001",
        "text": "In urban flood scenarios, prioritize evacuation of ground-floor and basement dwellings first. Rescue boats should target zones with water depth exceeding 1 meter.",
        "metadata": {"disaster_type": "flood", "region": "urban"},
    },
    {
        "id": "flood_002",
        "text": "For riverine flooding, establish evacuation routes on higher-elevation roads. Coordinate with local hospitals to pre-position emergency medical supplies before peak flood levels.",
        "metadata": {"disaster_type": "flood", "region": "riverine"},
    },
    {
        "id": "earthquake_001",
        "text": "After a major earthquake, search-and-rescue should focus on collapsed structures within the first 72 hours. Aftershock risk means rescue teams must assess structural stability before entry.",
        "metadata": {"disaster_type": "earthquake", "region": "any"},
    },
]

collection.add(
    ids=[p["id"] for p in sample_protocols],
    documents=[p["text"] for p in sample_protocols],
    metadatas=[p["metadata"] for p in sample_protocols],
)

In [5]:
results = collection.query(
    query_texts=["flood is hitting a city, what's the evacuation priority?"],
    n_results=2,
)
results

{'ids': [['flood_001', 'flood_002']],
 'embeddings': None,
 'documents': [['In urban flood scenarios, prioritize evacuation of ground-floor and basement dwellings first. Rescue boats should target zones with water depth exceeding 1 meter.',
   'For riverine flooding, establish evacuation routes on higher-elevation roads. Coordinate with local hospitals to pre-position emergency medical supplies before peak flood levels.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'region': 'urban', 'disaster_type': 'flood'},
   {'region': 'riverine', 'disaster_type': 'flood'}]],
 'distances': [[0.32835853099823, 0.39482027292251587]]}

In [6]:
results_filtered = collection.query(
    query_texts=["what should rescue teams do after a major quake?"],
    n_results=2,
    where={"disaster_type": "earthquake"},
)
results_filtered

{'ids': [['earthquake_001']],
 'embeddings': None,
 'documents': [['After a major earthquake, search-and-rescue should focus on collapsed structures within the first 72 hours. Aftershock risk means rescue teams must assess structural stability before entry.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'region': 'any', 'disaster_type': 'earthquake'}]],
 'distances': [[0.30614495277404785]]}

In [ ]:
from openai import OpenAI

llm_client = OpenAI()



def generate_grounded_plan(situation:str, disaster_type:str):
    retrieved = collection.query(
        query_texts = [situation],
        n_results = 2,
        where = {"disaster_type":  disaster_type},
    )

    context = "\n".join(retrieved["documents"][0])

    prompt = f"""Using ONLY the protocol context below, draft a short 3-phase response plan for this situation.
If the context doesn't cover something, say so instead of inventing it.

Situation: {situation}

Protocol context:
{context}

Return a brief 3-phase plan (Immediate / Short-term / Recovery)"""

    response = llm_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content, context

# this is the actual RAG pattern — retrieved text gets injected directly into the prompt, and the instruction "use ONLY the context" is what the Quality Checker will later verify against (did the plan actually stick to this, or invent resources not in used_context?). 
# This function is basically response_planner.py's real logic, just not in a file yet.

In [8]:
plan, used_context = generate_grounded_plan(
    "City flood, water rising fast in Zone A, hundreds trapped",
    disaster_type="flood",
)
print("PLAN:\n", plan)
print("\nGROUNDED IN:\n", used_context)

PLAN:
 **3-Phase Response Plan for City Flood in Zone A**

**Immediate Phase:**
- Initiate evacuation of residents trapped in ground-floor and basement dwellings in Zone A. Deploy rescue boats to areas where water depth exceeds 1 meter.

**Short-term Phase:**
- Establish evacuation routes using higher-elevation roads. Coordinate with local hospitals to ensure emergency medical supplies are pre-positioned in accessible locations before peak flood levels are reached.

**Recovery Phase:**
- Assess the damage in Zone A and provide support for displaced residents, including temporary shelters and resources for return to their homes once it is safe.

GROUNDED IN:
 In urban flood scenarios, prioritize evacuation of ground-floor and basement dwellings first. Rescue boats should target zones with water depth exceeding 1 meter.
For riverine flooding, establish evacuation routes on higher-elevation roads. Coordinate with local hospitals to pre-position emergency medical supplies before peak flood